**Description**
**action**: [K1_t, K2_t, K3_t, K4_t, K5_t, K6_t, d_t] (7D), K1-K6 are stiffness, d is damping factor, take the damping matrix to be [B1_t, B2_t, B3_t, B4_t, B5_t, B6_t] = d_t * sqrt(K_t)
**state**: [e_t, pd_t] (6D) tracking error e, desired position at every time step pd_t, the final goal is to reach the desired position-the hole and insert the peg in a fixed hole, which is located at [0, -0.7, 0.0]

**mission**: use adversial inverse reinforcement learning to train the reward function of this peg in hole task.

since we are using position control robot, we should use admittance control. The modified desired position pd_new_t = pd_t + e_t, and the desired orientation should stay the same, which means Rd_t is a constant. With these calculated, we can get the desired joint position q_d_t at every time step, and control the robot to reach the next state.

the reasons for choosing these state variables are:
- pd_t stands for the ideal trajectory
- e_t stands for the admittance dynamics, M * e'' + B * e' + K * e = F_external
by combining them, we can get the proper infomations for the robot to interact with the environment

and for the actions, K should be a 6 * 6 matrix, for simplicity, we consider it a diagonal matrix made of 6 components, so is B. The mass matrix is diagonal as well, and is given in advance.

下面是AIRL算法的总结，
Adversarial Inverse Reinforcement Learning, AIRL的流程可以总结为以下几个关键步骤
1. **初始化阶段**：
   - 获取专家轨迹数据集  $D = \{\tau_{E,1}, \tau_{E,2}, ..., \tau_{E,N}\}$ （步骤1）。
   - 初始化策略网络  $\pi$  和判别器  $D_{\theta,\phi}$ （步骤2）。判别器的结构如式(4)所示，包含奖励近似器  $g_\theta(s)$  和形状项  $h_\phi(s)$ 。
2. **迭代训练循环**（步骤3-8）：
   - **轨迹收集**：通过当前策略  $\pi$  在环境中执行，生成轨迹  $\tau_i = (s_0, a_0, ..., s_T, a_T)$ （步骤4）。
   - **判别器训练**：使用二元逻辑回归训练判别器  $D_{\theta,\phi}$ ，区分专家轨迹与策略生成的轨迹（步骤5）。判别器的输出形式为：
     $$
     D_{\theta,\phi}(s,a,s') = \frac{\exp(f_{\theta,\phi}(s,a,s'))}{\exp(f_{\theta,\phi}(s,a,s')) + \pi(a|s)},
     $$
     其中  $f_{\theta,\phi}(s,a,s') = g_\theta(s) + \gamma h_\phi(s') - h_\phi(s)$ 。
   - **奖励函数更新**：从判别器的输出中提取奖励函数（步骤6）：
     $$
     r_{\theta,\phi}(s,a,s') = \log D_{\theta,\phi}(s,a,s') - \log(1 - D_{\theta,\phi}(s,a,s')) 。
     $$
   - **策略优化**：使用任意策略优化方法（如TRPO）更新策略  $\pi$ ，以最大化从判别器导出的奖励（步骤7）。
3. **终止条件**：
   - 重复上述迭代直至达到预设的训练步数  $N$ 。
- **关键特性**：
- **对抗训练**：通过判别器与策略的博弈，逐步提升奖励函数和策略的性能（类似GAN框架）[1]。
- **动态鲁棒性**：通过设计  $f_{\theta,\phi}$  的结构（式4），AIRL学习的奖励  $g_\theta(s)$  能够与动力学解耦（Theorem 5.1-5.2），从而在环境动态变化时仍保持有效性。
- **高效性**：相比轨迹级IRL方法（如GAN-GCL），AIRL在状态-动作对级别操作，降低了方差并提升了可扩展性（Sec. 4）[1]。
- **输出结果**：
- 最终输出为学习到的策略  $\pi$  和奖励函数  $g_\theta(s)$

在我们的情景下，我会提供符合上面状态-动作对的专家轨迹，你需要在这个文件中实现上述的AIRL算法。环境文件peg_in_hole_env.py需要一些对应的更改，使用的mujoco模型文件为jaka_pih.xml,原来的专家轨迹生成器是vac_expert_generator.py,这个生成器在之后需要大改，你现在暂时不用关注。你需要做的就是按照上述的背景实现AIRL算法。

In [ ]:
#!/usr/bin/env python3
"""
AIRL (Adversarial Inverse Reinforcement Learning) Trainer for Variable Admittance Control
Peg-in-Hole Task Implementation

Variable Admittance Control:
- State Space: [e_t, pd_t] (6D) where e_t is tracking error, pd_t is desired position  
- Action Space: [K1, K2, K3, K4, K5, K6, d_t] (7D) impedance parameters
- Modified desired position: pd_new_t = pd_t + e_t
- Admittance dynamics: M * e'' + B * e' + K * e = F_external
- Damping matrix: B = d_t * sqrt(K)
"""

import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal
import pickle
import matplotlib.pyplot as plt
from collections import deque
import random
from typing import Dict, List, Tuple, Optional
import logging

# Add path for environment
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))
from envs.peg_in_hole_env import PegInHoleEnv

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Neural Network Components for AIRL

class RewardNetwork(nn.Module):
    """
    AIRL Reward Approximator g_θ(s)
    Maps state to reward scalar
    """
    def __init__(self, state_dim, hidden_dim=256):
        super(RewardNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, state):
        return self.network(state)

class ValueNetwork(nn.Module):
    """
    AIRL Value Function h_φ(s)
    Maps state to value scalar
    """
    def __init__(self, state_dim, hidden_dim=256):
        super(ValueNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, state):
        return self.network(state)

class AIRLDiscriminator(nn.Module):
    """
    AIRL Discriminator D_θ,φ(s,a,s')
    Combines reward and value networks
    """
    def __init__(self, state_dim, action_dim, hidden_dim=256, gamma=0.99):
        super(AIRLDiscriminator, self).__init__()
        self.reward_net = RewardNetwork(state_dim, hidden_dim)
        self.value_net = ValueNetwork(state_dim, hidden_dim)
        self.gamma = gamma
        
    def forward(self, state, action, next_state, log_prob):
        """
        Compute discriminator output
        D(s,a,s') = exp(f) / (exp(f) + π(a|s))
        where f = g(s) + γh(s') - h(s)
        """
        reward = self.reward_net(state)
        value_curr = self.value_net(state)
        value_next = self.value_net(next_state)
        
        f = reward + self.gamma * value_next - value_curr
        
        # AIRL discriminator formula
        exp_f = torch.exp(f)
        exp_log_prob = torch.exp(log_prob.unsqueeze(-1))
        
        discriminator_output = exp_f / (exp_f + exp_log_prob)
        
        return discriminator_output, reward
    
    def get_reward(self, state, action, next_state, log_prob):
        """Extract reward for RL training"""
        with torch.no_grad():
            disc_output, _ = self.forward(state, action, next_state, log_prob)
            # AIRL reward: log(D) - log(1-D)
            reward = torch.log(disc_output + 1e-8) - torch.log(1 - disc_output + 1e-8)
        return reward.squeeze()

In [ ]:
class VariableImpedancePolicy(nn.Module):
    """
    Policy Network for Variable Impedance Control
    Input: [e_t, pd_t] (6D) - tracking error and desired position
    Output: [K1, K2, K3, K4, K5, K6, d_t] (7D) - impedance parameters
    """
    def __init__(self, state_dim=6, action_dim=7, hidden_dim=256):
        super(VariableImpedancePolicy, self).__init__()
        
        self.actor_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Mean and log_std for continuous actions
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.orthogonal_(module.weight, gain=0.01)
            module.bias.data.zero_()
    
    def forward(self, state):
        features = self.actor_net(state)
        mean = self.mean_head(features)
        log_std = self.log_std_head(features)
        
        # Clamp log_std to reasonable range
        log_std = torch.clamp(log_std, -10, 2)
        
        return mean, log_std
    
    def sample_action(self, state):
        """Sample action from policy"""
        mean, log_std = self.forward(state)
        std = torch.exp(log_std)
        
        # Create normal distribution
        dist = Normal(mean, std)
        action = dist.sample()
        
        # Apply sigmoid to ensure actions are in [0, 1]
        action = torch.sigmoid(action)
        log_prob = dist.log_prob(action).sum(dim=-1)
        
        return action, log_prob
    
    def evaluate_action(self, state, action):
        """Evaluate log probability of given action"""
        mean, log_std = self.forward(state)
        std = torch.exp(log_std)
        
        # Inverse sigmoid to get pre-sigmoid action
        action_pre_sigmoid = torch.log(action / (1 - action + 1e-8))
        
        dist = Normal(mean, std)
        log_prob = dist.log_prob(action_pre_sigmoid).sum(dim=-1)
        
        return log_prob

In [ ]:
class ExpertDataLoader:
    """
    Load and process expert demonstration data
    Expected format: [e_t, pd_t, action] where:
    - e_t: tracking error (3D position)
    - pd_t: desired position (3D position) 
    - action: [K1, K2, K3, K4, K5, K6, d_t] impedance parameters
    """
    def __init__(self, expert_data_path):
        self.expert_data_path = expert_data_path
        self.trajectories = []
        self.load_expert_data()
    
    def load_expert_data(self):
        """Load expert trajectories from pickle file"""
        try:
            with open(self.expert_data_path, 'rb') as f:
                data = pickle.load(f)
            
            print(f"Loaded expert data from {self.expert_data_path}")
            
            # Convert to our format if needed
            if isinstance(data, dict):
                if 'observations' in data and 'actions' in data:
                    # IRL format from vac_expert_generator.py
                    observations = np.array(data['observations'])
                    actions = np.array(data['actions'])
                    
                    # Adapt observations to [e_t(3), pd_t(3)] format
                    # If observations are 12D [pose_error(6), velocity_error(6)],
                    # we take pose_error[:3] as e_t and create pd_t from trajectory
                    if observations.shape[1] == 12:
                        # Use position error as tracking error
                        tracking_errors = observations[:, :3]  # First 3D are position errors
                        
                        # Create desired positions (simplified - could be improved)
                        # For now, assume pd_t moves from start to target
                        start_pos = np.array([0.0, -0.5, 0.14])  # Start above hole
                        target_pos = np.array([0.0, -0.7, 0.01])  # Hole position
                        
                        N = len(tracking_errors)
                        t_values = np.linspace(0, 1, N)
                        desired_positions = np.array([start_pos + t * (target_pos - start_pos) for t in t_values])
                        
                        # Combined states [e_t, pd_t]
                        states = np.hstack([tracking_errors, desired_positions])
                    else:
                        # Assume data is already in correct format
                        states = observations
                    
                    # Create trajectory
                    trajectory = {
                        'states': states,
                        'actions': actions,
                        'next_states': states[1:] if len(states) > 1 else states
                    }
                    self.trajectories.append(trajectory)
                    
                    print(f"Processed trajectory with {len(states)} steps")
                    print(f"State dimension: {states.shape[1]}")
                    print(f"Action dimension: {actions.shape[1]}")
            
        except FileNotFoundError:
            print(f"Expert data file {self.expert_data_path} not found!")
            print("Please generate expert data first using vac_expert_generator.py")
        except Exception as e:
            print(f"Error loading expert data: {e}")
    
    def get_expert_transitions(self, batch_size=256):
        """Get random batch of expert state-action transitions"""
        if not self.trajectories:
            return None, None, None, None
        
        states, actions, next_states = [], [], []
        
        for _ in range(batch_size):
            # Sample random trajectory
            traj = random.choice(self.trajectories)
            
            # Sample random transition from trajectory
            if len(traj['states']) > 1:
                idx = random.randint(0, len(traj['states']) - 2)
                states.append(traj['states'][idx])
                actions.append(traj['actions'][idx])
                next_states.append(traj['states'][idx + 1])
            else:
                states.append(traj['states'][0])
                actions.append(traj['actions'][0])
                next_states.append(traj['states'][0])
        
        return (np.array(states), np.array(actions), 
                np.array(next_states), np.ones(batch_size))  # expert labels = 1

class ReplayBuffer:
    """Experience replay buffer for policy trajectories"""
    def __init__(self, max_size=100000):
        self.max_size = max_size
        self.buffer = deque(maxlen=max_size)
    
    def add(self, state, action, next_state, reward, done, log_prob):
        self.buffer.append((state, action, next_state, reward, done, log_prob))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, min(len(self.buffer), batch_size))
        states, actions, next_states, rewards, dones, log_probs = zip(*batch)
        
        return (np.array(states), np.array(actions), np.array(next_states),
                np.array(rewards), np.array(dones), np.array(log_probs))
    
    def __len__(self):
        return len(self.buffer)

In [ ]:
class AIRLTrainer:
    """
    AIRL (Adversarial Inverse Reinforcement Learning) Trainer
    for Variable Impedance Control
    """
    def __init__(self, 
                 env,
                 expert_data_path,
                 state_dim=6,
                 action_dim=7,
                 hidden_dim=256,
                 lr_policy=3e-4,
                 lr_discriminator=3e-4,
                 gamma=0.99,
                 device=None):
        
        self.env = env
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Initialize networks
        self.policy = VariableImpedancePolicy(state_dim, action_dim, hidden_dim).to(self.device)
        self.discriminator = AIRLDiscriminator(state_dim, action_dim, hidden_dim, gamma).to(self.device)
        
        # Optimizers
        self.policy_optimizer = optim.Adam(self.policy.parameters(), lr=lr_policy)
        self.discriminator_optimizer = optim.Adam(self.discriminator.parameters(), lr=lr_discriminator)
        
        # Data management
        self.expert_loader = ExpertDataLoader(expert_data_path)
        self.replay_buffer = ReplayBuffer()
        
        # Training metrics
        self.training_metrics = {
            'discriminator_loss': [],
            'policy_loss': [],
            'policy_reward': [],
            'discriminator_accuracy': [],
            'episode_rewards': []
        }
        
        print(f"AIRL Trainer initialized on {self.device}")
        print(f"State dim: {state_dim}, Action dim: {action_dim}")
    
    def collect_trajectories(self, num_episodes=10, max_steps=500):
        """Collect trajectories using current policy"""
        trajectories = []
        episode_rewards = []
        
        for episode in range(num_episodes):
            trajectory = []
            state, _ = self.env.reset()
            episode_reward = 0
            
            for step in range(max_steps):
                # Convert state to tensor
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                
                # Sample action from policy
                with torch.no_grad():
                    action, log_prob = self.policy.sample_action(state_tensor)
                    action = action.cpu().numpy()[0]
                    log_prob = log_prob.cpu().numpy()[0]
                
                # Take step in environment
                next_state, reward, done, truncated, info = self.env.step(action)
                
                # Store transition
                trajectory.append({
                    'state': state,
                    'action': action,
                    'next_state': next_state,
                    'reward': reward,
                    'done': done or truncated,
                    'log_prob': log_prob
                })
                
                # Add to replay buffer
                self.replay_buffer.add(state, action, next_state, reward, done or truncated, log_prob)
                
                state = next_state
                episode_reward += reward
                
                if done or truncated:
                    break
            
            trajectories.append(trajectory)
            episode_rewards.append(episode_reward)
            
            if episode % 5 == 0:
                print(f"Episode {episode}, Reward: {episode_reward:.2f}, Steps: {len(trajectory)}")
        
        self.training_metrics['episode_rewards'].extend(episode_rewards)
        return trajectories
    
    def train_discriminator(self, policy_batch_size=256, expert_batch_size=256):
        """Train AIRL discriminator to distinguish expert vs policy data"""
        
        # Get expert data
        expert_states, expert_actions, expert_next_states, expert_labels = \
            self.expert_loader.get_expert_transitions(expert_batch_size)
        
        if expert_states is None:
            print("No expert data available!")
            return 0.0
        
        # Get policy data
        if len(self.replay_buffer) < policy_batch_size:
            return 0.0
        
        policy_states, policy_actions, policy_next_states, _, _, policy_log_probs = \
            self.replay_buffer.sample(policy_batch_size)
        
        # Convert to tensors
        expert_states = torch.FloatTensor(expert_states).to(self.device)
        expert_actions = torch.FloatTensor(expert_actions).to(self.device)
        expert_next_states = torch.FloatTensor(expert_next_states).to(self.device)
        
        policy_states = torch.FloatTensor(policy_states).to(self.device)
        policy_actions = torch.FloatTensor(policy_actions).to(self.device)
        policy_next_states = torch.FloatTensor(policy_next_states).to(self.device)
        policy_log_probs = torch.FloatTensor(policy_log_probs).to(self.device)
        
        # Get expert log probs from current policy
        expert_log_probs = self.policy.evaluate_action(expert_states, expert_actions)
        
        # Forward pass through discriminator
        expert_disc_output, _ = self.discriminator(expert_states, expert_actions, 
                                                 expert_next_states, expert_log_probs)
        policy_disc_output, _ = self.discriminator(policy_states, policy_actions, 
                                                 policy_next_states, policy_log_probs)
        
        # Binary classification loss
        expert_loss = F.binary_cross_entropy(expert_disc_output.squeeze(), 
                                           torch.ones_like(expert_disc_output.squeeze()))
        policy_loss = F.binary_cross_entropy(policy_disc_output.squeeze(), 
                                           torch.zeros_like(policy_disc_output.squeeze()))
        
        discriminator_loss = expert_loss + policy_loss
        
        # Update discriminator
        self.discriminator_optimizer.zero_grad()
        discriminator_loss.backward()
        self.discriminator_optimizer.step()
        
        # Calculate accuracy
        expert_pred = (expert_disc_output > 0.5).float()
        policy_pred = (policy_disc_output < 0.5).float()
        accuracy = (expert_pred.sum() + policy_pred.sum()) / (expert_batch_size + policy_batch_size)
        
        self.training_metrics['discriminator_loss'].append(discriminator_loss.item())
        self.training_metrics['discriminator_accuracy'].append(accuracy.item())
        
        return discriminator_loss.item()
    
    def train_policy(self, batch_size=256):
        """Train policy using AIRL-derived rewards"""
        
        if len(self.replay_buffer) < batch_size:
            return 0.0
        
        # Sample transitions
        states, actions, next_states, _, _, old_log_probs = \
            self.replay_buffer.sample(batch_size)
        
        # Convert to tensors
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.FloatTensor(actions).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        old_log_probs = torch.FloatTensor(old_log_probs).to(self.device)
        
        # Get current policy log probs
        new_log_probs = self.policy.evaluate_action(states, actions)
        
        # Get AIRL rewards
        airl_rewards = self.discriminator.get_reward(states, actions, next_states, new_log_probs)
        
        # Simple policy gradient loss (can be replaced with PPO/TRPO)
        ratio = torch.exp(new_log_probs - old_log_probs)
        policy_loss = -torch.mean(ratio * airl_rewards)
        
        # Update policy
        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        self.policy_optimizer.step()
        
        self.training_metrics['policy_loss'].append(policy_loss.item())
        self.training_metrics['policy_reward'].append(torch.mean(airl_rewards).item())
        
        return policy_loss.item()
    
    def train(self, num_iterations=1000, 
              collect_episodes_per_iter=5, 
              discriminator_updates_per_iter=5,
              policy_updates_per_iter=5):
        """Main AIRL training loop"""
        
        print("Starting AIRL training...")
        
        for iteration in range(num_iterations):
            print(f"\n=== Iteration {iteration + 1}/{num_iterations} ===")
            
            # Collect trajectories using current policy
            trajectories = self.collect_trajectories(collect_episodes_per_iter)
            
            # Train discriminator
            disc_losses = []
            for _ in range(discriminator_updates_per_iter):
                disc_loss = self.train_discriminator()
                disc_losses.append(disc_loss)
            
            # Train policy
            policy_losses = []
            for _ in range(policy_updates_per_iter):
                policy_loss = self.train_policy()
                policy_losses.append(policy_loss)
            
            # Log progress
            avg_episode_reward = np.mean(self.training_metrics['episode_rewards'][-collect_episodes_per_iter:])
            avg_disc_loss = np.mean(disc_losses) if disc_losses else 0
            avg_policy_loss = np.mean(policy_losses) if policy_losses else 0
            avg_disc_acc = np.mean(self.training_metrics['discriminator_accuracy'][-discriminator_updates_per_iter:]) if self.training_metrics['discriminator_accuracy'] else 0
            
            print(f"Episode Reward: {avg_episode_reward:.3f}")
            print(f"Discriminator Loss: {avg_disc_loss:.4f}, Accuracy: {avg_disc_acc:.3f}")
            print(f"Policy Loss: {avg_policy_loss:.4f}")
            
            # Save models periodically
            if (iteration + 1) % 50 == 0:
                self.save_models(f"checkpoints/airl_iter_{iteration + 1}")
        
        print("Training completed!")
    
    def save_models(self, save_path):
        """Save trained models"""
        os.makedirs(save_path, exist_ok=True)
        
        torch.save(self.policy.state_dict(), f"{save_path}/policy.pt")
        torch.save(self.discriminator.state_dict(), f"{save_path}/discriminator.pt")
        
        # Save training metrics
        with open(f"{save_path}/metrics.pkl", 'wb') as f:
            pickle.dump(self.training_metrics, f)
        
        print(f"Models saved to {save_path}")
    
    def plot_training_metrics(self):
        """Plot training metrics"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Episode rewards
        axes[0, 0].plot(self.training_metrics['episode_rewards'])
        axes[0, 0].set_title('Episode Rewards')
        axes[0, 0].set_xlabel('Episode')
        axes[0, 0].set_ylabel('Reward')
        
        # Discriminator loss
        axes[0, 1].plot(self.training_metrics['discriminator_loss'])
        axes[0, 1].set_title('Discriminator Loss')
        axes[0, 1].set_xlabel('Update')
        axes[0, 1].set_ylabel('Loss')
        
        # Policy loss
        axes[1, 0].plot(self.training_metrics['policy_loss'])
        axes[1, 0].set_title('Policy Loss')
        axes[1, 0].set_xlabel('Update')
        axes[1, 0].set_ylabel('Loss')
        
        # Discriminator accuracy
        axes[1, 1].plot(self.training_metrics['discriminator_accuracy'])
        axes[1, 1].set_title('Discriminator Accuracy')
        axes[1, 1].set_xlabel('Update')
        axes[1, 1].set_ylabel('Accuracy')
        
        plt.tight_layout()
        plt.show()

In [ ]:
# Training Configuration and Execution

def main():
    """Main training function"""
    
    # Configuration
    config = {
        'expert_data_path': 'expert_pih_0.pkl',  # Path to expert data
        'state_dim': 6,  # [e_t(3), pd_t(3)]
        'action_dim': 7,  # [K1, K2, K3, K4, K5, K6, d_t]
        'hidden_dim': 256,
        'lr_policy': 3e-4,
        'lr_discriminator': 3e-4,
        'gamma': 0.99,
        'num_iterations': 200,
        'collect_episodes_per_iter': 5,
        'discriminator_updates_per_iter': 5,
        'policy_updates_per_iter': 5
    }
    
    print("=== AIRL Training for Variable Impedance Control ===")
    print(f"Configuration: {config}")
    
    # Create environment
    try:
        env = PegInHoleEnv(
            xml_path="models/jaka_zu12/jaka_pih.xml",
            control_dt=0.008,  # 125Hz
            max_episode_steps=500
        )
        print("Environment created successfully")
        print(f"Observation space: {env.observation_space.shape}")
        print(f"Action space: {env.action_space.shape}")
        
    except Exception as e:
        print(f"Error creating environment: {e}")
        return
    
    # Initialize trainer
    trainer = AIRLTrainer(
        env=env,
        expert_data_path=config['expert_data_path'],
        state_dim=config['state_dim'],
        action_dim=config['action_dim'],
        hidden_dim=config['hidden_dim'],
        lr_policy=config['lr_policy'],
        lr_discriminator=config['lr_discriminator'],
        gamma=config['gamma']
    )
    
    # Check if expert data is available
    if not trainer.expert_loader.trajectories:
        print("\n" + "="*50)
        print("WARNING: No expert data found!")
        print("Please run the expert data generation first:")
        print("cd script/vac")
        print("python vac_expert_generator.py")
        print("="*50)
        return
    
    # Create checkpoints directory
    os.makedirs("checkpoints", exist_ok=True)
    
    # Start training
    try:
        trainer.train(
            num_iterations=config['num_iterations'],
            collect_episodes_per_iter=config['collect_episodes_per_iter'],
            discriminator_updates_per_iter=config['discriminator_updates_per_iter'],
            policy_updates_per_iter=config['policy_updates_per_iter']
        )
        
        # Plot results
        trainer.plot_training_metrics()
        
        # Save final models
        trainer.save_models("checkpoints/final")
        
    except KeyboardInterrupt:
        print("\nTraining interrupted by user")
        trainer.save_models("checkpoints/interrupted")
    except Exception as e:
        print(f"Training error: {e}")
        trainer.save_models("checkpoints/error")
    
    print("Training completed!")

# Example usage - uncomment to run
# if __name__ == "__main__":
#     main()

In [ ]:
# Test the Implementation

# First, let's test our components independently
print("Testing AIRL Implementation Components...")

# Test 1: Neural Networks
print("\n1. Testing Neural Networks...")
try:
    state_dim, action_dim = 6, 7
    
    # Test policy network
    policy = VariableImpedancePolicy(state_dim, action_dim)
    dummy_state = torch.randn(1, state_dim)
    action, log_prob = policy.sample_action(dummy_state)
    print(f"✓ Policy network: State {dummy_state.shape} -> Action {action.shape}, Log prob {log_prob.shape}")
    
    # Test discriminator
    discriminator = AIRLDiscriminator(state_dim, action_dim)
    dummy_next_state = torch.randn(1, state_dim)
    disc_output, reward = discriminator(dummy_state, action, dummy_next_state, log_prob)
    print(f"✓ Discriminator: Output {disc_output.shape}, Reward {reward.shape}")
    
except Exception as e:
    print(f"✗ Neural network test failed: {e}")

# Test 2: Environment
print("\n2. Testing Environment...")
try:
    env = PegInHoleEnv(xml_path="models/jaka_zu12/jaka_pih.xml")
    obs, info = env.reset()
    print(f"✓ Environment reset: Observation shape {obs.shape}")
    print(f"✓ Observation space: {env.observation_space.shape}")
    print(f"✓ Action space: {env.action_space.shape}")
    
    # Test step
    action = np.random.rand(7)
    next_obs, reward, done, truncated, info = env.step(action)
    print(f"✓ Environment step: Action {action.shape} -> Next obs {next_obs.shape}")
    
    # Check observation format
    print(f"✓ Observation format: [e_t(3), pd_t(3)] = {obs}")
    env.close()
    
except Exception as e:
    print(f"✗ Environment test failed: {e}")

# Test 3: Expert Data Loader (if data exists)
print("\n3. Testing Expert Data Loader...")
try:
    expert_loader = ExpertDataLoader('expert_pih_0.pkl')
    if expert_loader.trajectories:
        states, actions, next_states, labels = expert_loader.get_expert_transitions(batch_size=32)
        print(f"✓ Expert data loaded: {len(expert_loader.trajectories)} trajectories")
        print(f"✓ Batch sampling: States {states.shape}, Actions {actions.shape}")
    else:
        print("⚠ No expert data found - this is expected if not generated yet")
        
except Exception as e:
    print(f"⚠ Expert data loader: {e}")

print("\n4. Components ready for training!")
print("Next steps:")
print("- Generate expert data using: python vac_expert_generator.py")  
print("- Run main() function to start AIRL training")
print("- Monitor training progress with plot_training_metrics()")

# Display summary
print(f"\n{'='*50}")
print("AIRL IMPLEMENTATION SUMMARY")
print(f"{'='*50}")
print("✓ State Space: [e_t(3), pd_t(3)] = 6D tracking error + desired position")
print("✓ Action Space: [K1, K2, K3, K4, K5, K6, d_t] = 7D impedance parameters") 
print("✓ Admittance Control: M*e'' + B*e' + K*e = F_external")
print("✓ Damping Matrix: B = d_t * sqrt(K)")
print("✓ AIRL Discriminator: D = exp(f) / (exp(f) + π(a|s))")
print("✓ Reward Function: r = log(D) - log(1-D)")
print("✓ Policy Network: Neural network with continuous action sampling")
print("✓ Training Loop: Adversarial training between policy and discriminator")
print("✓ Variable Admittance: pd_new = pd + e_admittance")
print("✓ Tracking Error: e_t = p - pd (actual - desired position)")
print(f"{'='*50}")